# Re-OCR and Monthly Intake

> **Sanitized release.** This notebook is a faithful copy of the original working
> notebook from a federal document-classification project, edited for public
> sharing: cell outputs are stripped, file paths and system names are
> generalized, and the ten real business categories are renamed `CAT-A` … `CAT-J`.
> Cells that were duplicated once per category in the original are collapsed to
> one or two representative copies, marked with a note. Nothing else about the
> structure or approach has been changed.


Two halves:

1. **Re-OCR** *(faithful)* — regenerate searchable PDFs for documents whose
   original text layer was unusable: render each page image at high
   resolution, OCR it with Tesseract, and merge the per-page results back
   into a single searchable PDF that mirrors the source directory tree.
2. **Monthly intake detection** *(reconstructed)* — the snapshot-diff script
   that fed this queue was not recovered; the cells in that section are a
   faithful reconstruction of its logic from the workflow it drove.

In [ ]:
import os
import regex
import PyPDF2 as PyPDF3
from PyPDF2 import PdfFileMerger, PdfFileReader
import traceback
import numpy as np
import pandas as pd
from tqdm import tqdm
import cv2                                       #supported image-preprocessing experiments
import pytesseract
from imutils.perspective import four_point_transform
import fitz

## Re-OCR: regenerate searchable PDFs

In [ ]:
def reOCR():
    pdf_dirs_csv = pd.read_csv('reprocess_queue.csv')#open dir csv
    pdf_dirs = pdf_dirs_csv['directory'].tolist()#send to list
    pdf_id = pdf_dirs_csv['document_id'].tolist()#send to list
    pdf_dirs_csv = None

    #call tesseract instance
    pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"

    #open pdf, search all pages, append results to list
    for ID, item in tqdm(zip(pdf_id, pdf_dirs)):

        dir = item

        try:
            #load pdf to memory
            pdf = PyPDF3.PdfFileReader(item, strict=False)
            doc = fitz.open(dir)

            #get total page numbers
            numPages = pdf.getNumPages()

            #walk through each page and generate temp images
            #(6x zoom matrix ~= 430 DPI render — the sweet spot found in the DPI tradeoff tests)
            temp_img_lst = []
            for page_num in range(0, numPages, 1):
                page = doc.loadPage(page_num)#read specific page
                pix = page.getPixmap(matrix=(fitz.Matrix(6, 6)))#get pixel map of page
                filename = '{}{}{}'.format('temp_img_', page_num, '.png')
                temp_img_lst.append(filename)#send filename to temp lst
                pix.writePNG(filename)#write page to png, and save

            #ocr each image and generate a corresponding pdf
            temp_pdf_lst = []
            for img in temp_img_lst:
                pdf = pytesseract.image_to_pdf_or_hocr(img, extension='pdf')#generate pdfs for each page
                pdf_filename = str(img[:-4] + '.pdf')
                with open(pdf_filename, 'wb') as f:
                    f.write(pdf)
                f.close()
                temp_pdf_lst.append(pdf_filename)
                os.remove(img)#remove image file

            #concat all pdfs into one
            merger = PdfFileMerger(strict=True)
            for p in temp_pdf_lst:
                merger.append(PdfFileReader(p))
                os.remove(p)

            #prepare filename string
            dir_str = str(dir)
            dir_split = dir_str.split('/')
            dir_split_last = dir_split[-1]

            #save in a mirrored output tree under reocr_output/
            out_dir = os.path.join('reocr_output', *dir_split[1:-1])
            try:
                os.makedirs(out_dir)
            except:
                pass
            #write everything to one main file
            merger.write(out_dir + '/' + dir_split_last)

        except Exception:
            traceback.print_exc()
            print("failed: " + dir)
    return print('done.')

Throughput from the original run of this cell: a **135-document batch of
full-length records finished in about 1 h 33 m — roughly 42 seconds per
document** at the 6× render zoom. Fast enough that a month's incoming
documents were never a backlog.

In [ ]:
reOCR()

## Monthly intake detection — *reconstructed*

> The original snapshot-diff script was not recovered for this release. The
> logic below is a reconstruction of what it did: each month, index the
> repository, diff against the previous month's index, and route anything new
> into the same OCR → scoring → review-pool machinery. It produced the
> `reprocess_queue.csv` consumed above (for documents needing re-OCR) and the
> new-document list for model scoring.

In [ ]:
#RECONSTRUCTED — original not recovered; logic matches the operated workflow.
import os
import pandas as pd

def snapshot_repository_index(repository_root):
    #walk the repository and index every document with its id, path, and mtime
    rows = []
    for root, _dirs, files in os.walk(repository_root):
        for name in files:
            if name.lower().endswith('.pdf'):
                path = os.path.join(root, name)
                rows.append({
                    'document_id': derive_document_id(path),
                    'directory': path,
                    'modified': os.path.getmtime(path),
                })
    return pd.DataFrame(rows)

def detect_new_documents(current, previous):
    #anything in this month's index that last month's index didn't have
    known = set(previous['document_id'])
    return current[~current['document_id'].isin(known)]

current_snapshot = snapshot_repository_index('repository_root/')
previous_snapshot = pd.read_csv('snapshots/previous_month.csv')

new_documents = detect_new_documents(current_snapshot, previous_snapshot)
print('|INFO| new documents this month: {}'.format(len(new_documents)))

#route into the pipeline:
# 1. documents without a usable text layer -> reprocess_queue.csv (re-OCR above)
# 2. all new documents -> OCR text extraction -> scoring across all 20 models
# 3. scored rows appended to the administrators' review pool
new_documents.to_csv('snapshots/new_documents_this_month.csv', index=False)
current_snapshot.to_csv('snapshots/previous_month.csv', index=False)

Run on a monthly cadence, this closed the loop: the corpus never went stale,
newly added documents were OCR'd (or re-OCR'd), scored by the same 20
classifiers, and ranked into the administrators' review pool — an operational
workflow rather than a one-time analysis.